In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

In [ ]:
m = 1.2              # kg
g = 9.81             # m/s^2
l = 0.22             # m, arm length
I_x = 0.015          # kg m^2
I_y = 0.015          # kg m^2
I_z = 0.028          # kg m^2
J = np.diag([I_x, I_y, I_z])

c_t = 1.8e-5         # thrust coefficient: T_i = c_t * q_i^2
c_r = 2.5e-7         # yaw drag torque coefficient

# Hover rotor speed for reference (all four equal)
q_hover = np.sqrt(m * g / (4 * c_t))
v_lin = 2.0       # constant forward trim speed (m/s)
k_drag = 0.18     # quadratic drag coefficient (N / (m/s)^2)
d_ref = 5.0       # target distance ahead at trim (m)
v_shot = 250.     # shooting speed in meters per second


In [ ]:
def target_fn(t, x):
    P0 = np.array([8.0, 1.5, 0.5])
    V0 = np.array([0.8, -0.2, 0.1])
    a0 = np.zeros(3)
    P_F = P0 + V0 * t
    V_F = V0
    a_F = a0
    return P_F, V_F, a_F

def skew(w):
    p, q, r = w
    return np.array([
        [0.0, -r,   q],
        [r,   0.0, -p],
        [-q,  p,   0.0]
    ])

def R_body_to_world(phi, theta, psi):
    cph, sph = np.cos(phi), np.sin(phi)
    cth, sth = np.cos(theta), np.sin(theta)
    cps, sps = np.cos(psi), np.sin(psi)

    return np.array([
        [cps*cth,  cps*sth*sph - sps*cph,  cps*sth*cph + sps*sph],
        [sps*cth,  sps*sth*sph + cps*cph,  sps*sth*cph - cps*sph],
        [-sth,     cth*sph,                cth*cph]
    ])

def euler_rate_matrix(phi, theta):
    cph, sph = np.cos(phi), np.sin(phi)
    cth = np.cos(theta)
    tth = np.tan(theta)

    return np.array([
        [1.0, sph*tth, cph*tth],
        [0.0, cph,    -sph],
        [0.0, sph/cth, cph/cth]
    ])

# ----------------------------
# Trim helpers for forward flight with quadratic drag
# ----------------------------
def trim_pitch_from_speed(v):
    return np.arctan2(k_drag * v**2, m * g)

def trim_thrust_from_speed(v):
    return np.hypot(m * g, k_drag * v**2)

def trim_u_from_speed(v):
    # u_i = q_i^2, equal on all four rotors at the trim thrust
    return trim_thrust_from_speed(v) / (4 * c_t)

def trim_rotor_speed_from_speed(v):
    return np.sqrt(trim_u_from_speed(v))

# Example rotor input function: hold the trim for v_lin
def rotor_speeds_fn(t, x):
    q0 = trim_rotor_speed_from_speed(v_lin)
    return np.full(4, q0)

# ----------------------------
# Nonlinear 18-state RHS with quadratic drag
# x = [P(3), V(3), angles(3), omega(3), rho(3), nu(3)]
# ----------------------------
def quad_target_rhs(t, x, rotor_speeds_fn, target_fn):
    P = x[0:3]
    V = x[3:6]
    phi, theta, psi = x[6:9]
    omega = x[9:12]
    rho = x[12:15]
    nu = x[15:18]

    R = R_body_to_world(phi, theta, psi)
    E = euler_rate_matrix(phi, theta)

    q_rot = np.asarray(rotor_speeds_fn(t, x), dtype=float)
    u = q_rot**2
    P_F, V_F, a_F = target_fn(t, x)

    T = c_t * np.sum(u)

    tau_phi   = l * c_t * (-u[0] - u[3] + u[1] + u[2])
    tau_theta = l * c_t * ( u[0] + u[1] - u[2] - u[3])
    tau_psi   = c_r * ( u[0] - u[1] + u[2] - u[3])
    tau = np.array([tau_phi, tau_theta, tau_psi])

    e3 = np.array([0.0, 0.0, 1.0])

    # Quadratic drag in world frame: F_drag = -k_drag * ||V|| * V
    Vnorm = np.linalg.norm(V)
    F_drag = -k_drag * Vnorm * V

    P_dot = V
    V_dot = (T / m) * (R @ e3) - g * e3 + F_drag / m

    euler_dot = E @ omega
    omega_dot = np.linalg.solve(J, tau - np.cross(omega, J @ omega))

    Omega = skew(omega)
    rho_dot = nu - Omega @ rho
    nu_dot = -Omega @ nu + R.T @ (a_F - V_dot)

    return np.concatenate([
        P_dot,
        V_dot,
        euler_dot,
        omega_dot,
        rho_dot,
        nu_dot
    ])

In [ ]:
def linearized_AB_18(v=v_lin, d=d_ref):
    theta0 = trim_pitch_from_speed(v)
    T0 = trim_thrust_from_speed(v)

    s = np.sin(theta0)
    c = np.cos(theta0)
    sec = 1.0 / c
    tan = np.tan(theta0)

    R0 = R_body_to_world(0.0, theta0, 0.0)
    RT0 = R0.T
    e3 = np.array([0.0, 0.0, 1.0])

    # State indices
    Px, Py, Pz = 0, 1, 2
    Vx, Vy, Vz = 3, 4, 5
    phi, theta, psi = 6, 7, 8
    p, q, r = 9, 10, 11
    rx, ry, rz = 12, 13, 14
    nux, nuy, nuz = 15, 16, 17

    A = np.zeros((18, 18))
    B = np.zeros((18, 4))

    # P_dot = V
    A[Px:Px+3, Vx:Vx+3] = np.eye(3)

    # Drag Jacobian at V* = [v,0,0], v>0:
    # d/dV ( -k ||V|| V / m ) = -(k/m) diag(2v, v, v)
    J_drag = -(k_drag / m) * np.diag([2.0 * v, v, v])

    # d/d(angles) of translational acceleration (T/m) R e3 at trim
    # evaluated at phi=0, theta=theta0, psi=0
    dacc_dangles = np.array([
        [0.0,            (T0 / m) * c,  0.0],
        [-(T0 / m),      0.0,           (T0 / m) * s],
        [0.0,           -(T0 / m) * s,  0.0],
    ])

    # V_dot block
    A[Vx:Vx+3, Vx:Vx+3] = J_drag
    A[Vx:Vx+3, phi:phi+3] = dacc_dangles

    # Input enters V_dot through total thrust only
    thrust_dir = R0 @ e3                      # = [sin(theta0), 0, cos(theta0)]
    B[Vx:Vx+3, :] = (c_t / m) * np.outer(thrust_dir, np.ones(4))

    # Euler kinematics at trim phi=0, theta=theta0
    # [phi_dot, theta_dot, psi_dot]^T = E0 [p,q,r]^T
    E0 = np.array([
        [1.0, 0.0, tan],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, sec],
    ])
    A[phi:phi+3, p:p+3] = E0

    # Rotational dynamics at omega*=0
    B[p, :] = l * c_t * np.array([-1, +1, +1, -1]) / I_x
    B[q, :] = l * c_t * np.array([+1, +1, -1, -1]) / I_y
    B[r, :] = c_r   * np.array([+1, -1, +1, -1]) / I_z

    # rho_dot = nu - omega x rho, with rho* = [d,0,0]
    A[rx:rx+3, nux:nux+3] = np.eye(3)
    A[ry, r] = -d
    A[rz, q] = +d

    # nu_dot = -R0^T * delta(V_dot)   at trim, since nu*=0 and a_F*=0
    A[nux:nux+3, Vx:Vx+3] = -RT0 @ J_drag
    A[nux:nux+3, phi:phi+3] = -RT0 @ dacc_dangles
    B[nux:nux+3, :] = -RT0 @ B[Vx:Vx+3, :]

    # Trim state and input
    x_trim = np.zeros(18)
    x_trim[Vx] = v
    x_trim[theta] = theta0
    x_trim[rx] = d

    u_trim = np.full(4, trim_u_from_speed(v))

    return A, B, x_trim, u_trim